#Case E-commerce Olist
Este desafio propõe a construção de um relatório executivo voltado a investidores e acionistas do setor de e-commerce, baseado no Brazilian E-Commerce. O objetivo é transformar dados transacionais em uma narrativa clara sobre desempenho comercial, eficiência logística e/ou satisfação do cliente, culminando em recomendações acionáveis e, quando possível, previsões fundamentadas.

O notebook analisa a base de dados da Olist com foco em duas perguntas norteadoras

1. **O crescimento da Olist é sustentável?**
* Receita, volume, ticket médio;

2. **O que está travando potencial de crescimento?**
* Pedidos não entregues, atrasos, insatisfação por região.


##Prolegômeno


**Base de dados:** `dataset_osquinze_wcordeir.csv`  
**Período:** 2016–2018  
**Linhas:** ~92.500 registros

In [23]:
#Libs = bibliotecas importantes

#manipulação dos dados
import pandas as pd
import numpy as np

#visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

# Carregando a base
df = pd.read_csv('dataset_osquinze_wcordeir.csv')
df['data_compra'] = pd.to_datetime(df['data_compra'])
df['data_entrega'] = pd.to_datetime(df['data_entrega'])

print('Shape:', df.shape)
print('\nColunas:', df.columns.tolist())


Shape: (92545, 18)

Colunas: ['data_compra', 'data_entrega', 'anomes_referencia', 'segmento_produto', 'uf_cliente', 'tipo_fluxo_logistico', 'status_pedido', 'metodo_pagamento', 'performance_entrega', 'likert', 'momento_likert', 'detalhes_likert', 'volume_pedidos', 'volume_clientes', 'valor_produtos', 'valor_frete', 'faturamento_total', 'nota_media_satisfacao']


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92545 entries, 0 to 92544
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   data_compra            92545 non-null  datetime64[ns]
 1   data_entrega           90832 non-null  datetime64[ns]
 2   anomes_referencia      92545 non-null  int64         
 3   segmento_produto       92545 non-null  object        
 4   uf_cliente             92545 non-null  object        
 5   tipo_fluxo_logistico   92545 non-null  object        
 6   status_pedido          92545 non-null  object        
 7   metodo_pagamento       92544 non-null  object        
 8   performance_entrega    92545 non-null  object        
 9   likert                 92545 non-null  object        
 10  momento_likert         92545 non-null  object        
 11  detalhes_likert        92545 non-null  object        
 12  volume_pedidos         92545 non-null  int64         
 13  v

In [25]:
# Convertendo colunas de data
df['data_compra'] = pd.to_datetime(df['data_compra'])
df['data_entrega'] = pd.to_datetime(df['data_entrega'])


## Pergunta 1: O crescimento da Olist é sustentável?

Para responder, vamos analisar três indicadores mês a mês:
- **Volume de pedidos** > a Olist está vendendo mais?
- **Faturamento total** > a receita acompanha o crescimento?
- **Ticket médio** > o valor por pedido está crescendo, caindo ou estável?

In [26]:
# Agrupando por mês de referência
crescimento = df.groupby('anomes_referencia').agg(
    volume_pedidos   = ('volume_pedidos',   'sum'),
    faturamento_total = ('faturamento_total', 'sum')
).reset_index()

# Calculando ticket médio
crescimento['ticket_medio'] = crescimento['faturamento_total'] / crescimento['volume_pedidos']

print(crescimento.head(10))

   anomes_referencia  volume_pedidos  faturamento_total  ticket_medio
0             201609               2             279.69    139.845000
1             201610             291           51354.52    176.476014
2             201612               1              19.62     19.620000
3             201701             787          136943.46    174.006938
4             201702            1724          283561.69    164.478939
5             201703            2628          425617.96    161.955084
6             201704            2389          405848.61    169.882214
7             201705            3660          582710.83    159.210609
8             201706            3220          499652.24    155.171503
9             201707            3975          578753.73    145.598423


### Gráfico 1: Evolução mensal de volume, faturamento e ticket médio

In [32]:
# Agrupamento
crescimento = df.groupby('anomes_referencia').agg(
    volume_pedidos    = ('volume_pedidos',    'sum'),
    faturamento_total = ('faturamento_total', 'sum')
).reset_index()

crescimento['ticket_medio'] = crescimento['faturamento_total'] / crescimento['volume_pedidos']

# Removendo meses incompletos
crescimento_limpo = crescimento[crescimento['anomes_referencia'] < 201809].copy()
crescimento_limpo['mes_ano'] = pd.to_datetime(
    crescimento_limpo['anomes_referencia'].astype(str), format='%Y%m'
)

# Gráfico com 3 subplots empilhados
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Volume de Pedidos', 'Faturamento Total (R$)', 'Ticket Médio (R$)'),
    shared_xaxes=True,
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(
    x=crescimento_limpo['mes_ano'], y=crescimento_limpo['volume_pedidos'],
    mode='lines+markers', name='Volume', line=dict(color='steelblue')),
    row=1, col=1)

fig.add_trace(go.Scatter(
    x=crescimento_limpo['mes_ano'], y=crescimento_limpo['faturamento_total'],
    mode='lines+markers', name='Faturamento', line=dict(color='seagreen')),
    row=2, col=1)

fig.add_trace(go.Scatter(
    x=crescimento_limpo['mes_ano'], y=crescimento_limpo['ticket_medio'],
    mode='lines+markers', name='Ticket Médio', line=dict(color='coral')),
    row=3, col=1)

fig.update_layout(
    height=700,
    title_text='Pergunta 1: Evolução Mensal da Olist',
    showlegend=False
)

fig.show()

Através dos dados apresentados, fica evidente que volume e faturamento demonstram crescimento consistente, ou seja, a Olist está escalando! Em janeiro de 2017 iniciamos o ano com 137mil reais. Porém, apenas 12 meses depois, em janeiro de 2018, haviamos **superado esse número 10x**, iniciando o ano com uma receita de ~1.1milhão

Vale ressaltar o pico em outubro de 2017, que coincide com época de Black Friday.

Ainda, o ticket médio se mantém estável (oscila emtre RS140,00 - RS180,00) e isso é extretamente positivo, pois significa que crescemos em volume sem precisar abaixar o preço para atrair clientes.

**Observação:** Dezembro de 2016 apresenta valor atípico (RS19) devido ao início das operações: apenas 1 pedido registrado naquele mês, insuficiente para representar uma tendência.

Logo, conseguimos responder a primeira pergunta! Sim, **o crescimento foi sustentável**. **Volume e faturamneto cresceram consitentemente e o ticket médio se manteve estável, sem sinais de deterioração de valor.**

## Pergunta 2: O que está travando o potencial de crescimento?

Vamos investigar quatro ângulos:
- **Nota de satisfação** > quem comprou, está satisfeito? Se não, por que?
- **Pedidos não entregues** > qual o volume e status?
- **Atrasos** > qual a taxa de pedidos fora do prazo?
- **Insatisfação por região** > onde a experiência é pior?

### Onde essa insatisfação acontece?
Identificando a correlação direta entre prazo de entrega e satisfação do cliente.


In [42]:
# --- Preparação: distribuição de notas ---
scores = df.groupby('likert')['volume_pedidos'].sum().reset_index()
scores.columns = ['nota', 'quantidade']

ordem_likert = ['Pessima', 'Ruim', 'Boa', 'Otima']
cores_likert = {
    'Pessima': '#cc3300',
    'Ruim':    '#FFA500',
    'Boa':     'lightgreen',
    'Otima':   'green'
}

fig1 = px.bar(
    scores,
    x='nota', y='quantidade',
    color='nota',
    color_discrete_map=cores_likert,
    category_orders={'nota': ordem_likert},
    title='Distribuição de Avaliações'
)
fig1.update_layout(showlegend=False)

# No fig1
fig1.update_layout(showlegend=False, bargap=0.6)

fig1.show()

# --- Preparação: nota por performance de entrega ---
media_performance = df.groupby('performance_entrega')['nota_media_satisfacao'].mean().reset_index()
media_performance.columns = ['performance', 'nota_media']
media_performance['nota_media'] = media_performance['nota_media'].round(2)

cores_performance = {
    'No Prazo':   'seagreen',
    'Em Transito': 'steelblue',
    'Atrasado':   '#cc3300'
}

fig2 = px.bar(
    media_performance,
    x='performance', y='nota_media',
    color='performance',
    color_discrete_map=cores_performance,
    text='nota_media',
    title='Nota Média por Performance de Entrega'
)
fig2.update_traces(textposition='outside')
fig2.update_layout(showlegend=False)

# No fig2
fig2.update_layout(showlegend=False, bargap=0.8)

fig2.show()

A satisfação dos clientes apresenta padrão bimodal, ou seja, dois picos. 58% os clientes atribuíram nota máxima (5), enquanto 11% deram nota mínima (1) indicando que, apesar da maioria satisfeita, existe uma parcela relevante com experiência negativa que representa oportunidade de melhoria.

Ainda, existe correlação direta entre prazo de entrega e satisfação do cliente. Pedidos entregues no prazo recebem média de 4.2, enquanto atrasos moderados despencam para 1.8.

In [28]:
# Distribuição de status dos pedidos
status = df.groupby('status_pedido')['volume_pedidos'].sum().reset_index()
status = status.sort_values('volume_pedidos', ascending=False)

print(status)
print(f'\nTotal de pedidos: {status["volume_pedidos"].sum():,}')

      status_pedido  volume_pedidos
2          Entregue           97190
3           Enviado            1109
4          Faturado             313
1  Em Processamento             302
0          Aprovado               2

Total de pedidos: 98,916


### Insatisfação por Região
Identificando os estados com pior experiência do cliente,
cruzando nota média de satisfação e taxa de atraso por UF.

In [30]:
# Agrupando por UF
regioes = df.groupby('uf_cliente').agg(
    nota_media        = ('nota_media_satisfacao', 'mean'),
    volume_pedidos    = ('volume_pedidos', 'sum'),
    pedidos_atrasados = ('volume_pedidos', lambda x: x[df.loc[x.index, 'performance_entrega'] == 'Atrasado'].sum())
).reset_index()

# Taxa de atraso por UF
regioes['taxa_atraso'] = regioes['pedidos_atrasados'] / regioes['volume_pedidos'] * 100

# Ordenando pelos piores (menor nota)
regioes_sorted = regioes.sort_values('nota_media')

print(regioes_sorted[['uf_cliente', 'nota_media', 'taxa_atraso', 'volume_pedidos']].head(10))

   uf_cliente  nota_media  taxa_atraso  volume_pedidos
21         RR    3.644444    11.111111              45
1          AL    3.761084    23.114355             411
9          MA    3.776567    19.079838             739
24         SE    3.833333    14.782609             345
13         PA    3.846073    12.012320             974
18         RJ    3.869415    13.016965           12791
4          BA    3.875266    13.668851            3358
5          CE    3.875667    14.759036            1328
16         PI    3.930328    15.415822             493
15         PE    4.014241    10.449575            1646


In [33]:
# Status dos pedidos
status = df.groupby('status_pedido')['volume_pedidos'].sum().reset_index()
status = status.sort_values('volume_pedidos', ascending=True)

# Performance de entrega
performance = df.groupby('performance_entrega')['volume_pedidos'].sum().reset_index()
performance = performance.sort_values('volume_pedidos', ascending=True)
total = performance['volume_pedidos'].sum()
performance['percentual'] = (performance['volume_pedidos'] / total * 100).round(1)
performance['label'] = performance.apply(
    lambda r: f"{r['volume_pedidos']:,} ({r['percentual']}%)", axis=1
)

# Cores por categoria
cores_performance = {
    'No Prazo': 'seagreen',
    'Atrasado': 'coral',
    'Em Transito': 'steelblue'
}

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Status dos Pedidos', 'Performance de Entrega'))

fig.add_trace(go.Bar(
    x=status['volume_pedidos'],
    y=status['status_pedido'],
    orientation='h',
    marker_color='steelblue',
    text=status['volume_pedidos'].apply(lambda x: f'{x:,}'),
    textposition='outside',
    name='Status'),
    row=1, col=1)

fig.add_trace(go.Bar(
    x=performance['volume_pedidos'],
    y=performance['performance_entrega'],
    orientation='h',
    marker_color=[cores_performance.get(p, 'gray') for p in performance['performance_entrega']],
    text=performance['label'],
    textposition='outside',
    name='Performance'),
    row=1, col=2)

fig.update_layout(
    height=400,
    title_text='Pergunta 2: Pedidos Não Entregues e Atrasos',
    showlegend=False
)

fig.show()

In [35]:
# Agrupando por UF
regioes = df.groupby('uf_cliente').agg(
    nota_media        = ('nota_media_satisfacao', 'mean'),
    volume_pedidos    = ('volume_pedidos', 'sum'),
    pedidos_atrasados = ('volume_pedidos', lambda x: x[df.loc[x.index, 'performance_entrega'] == 'Atrasado'].sum())
).reset_index()

regioes['taxa_atraso'] = (regioes['pedidos_atrasados'] / regioes['volume_pedidos'] * 100).round(1)
regioes['nota_media'] = regioes['nota_media'].round(2)

top10_piores = regioes.sort_values('nota_media').head(10)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('10 Estados com Pior Satisfação', 'Satisfação x Taxa de Atraso por Estado'))

# Barras horizontais — 10 piores
media_geral = regioes['nota_media'].mean()

fig.add_trace(go.Bar(
    x=top10_piores['nota_media'],
    y=top10_piores['uf_cliente'],
    orientation='h',
    marker_color='coral',
    text=top10_piores['nota_media'],
    textposition='outside',
    name='Nota Média'),
    row=1, col=1)

fig.add_vline(x=media_geral, line_dash='dash', line_color='gray',
              annotation_text=f'Média: {media_geral:.2f}', row=1, col=1)

# Dispersão — todos os estados
fig.add_trace(go.Scatter(
    x=regioes['taxa_atraso'],
    y=regioes['nota_media'],
    mode='markers+text',
    marker=dict(
        size=regioes['volume_pedidos'] / 500,
        color='steelblue',
        opacity=0.6
    ),
    text=regioes['uf_cliente'],
    textposition='top center',
    name='Estados'),
    row=1, col=2)

fig.update_xaxes(title_text='Nota Média', row=1, col=1)
fig.update_xaxes(title_text='Taxa de Atraso (%)', row=1, col=2)
fig.update_yaxes(title_text='Nota Média de Satisfação', row=1, col=2)

fig.update_layout(
    height=500,
    title_text='Pergunta 2: Insatisfação por Região',
    showlegend=False
)

fig.show()

Podemos perceber que,um dos motivos que pode estar travando o potencial de crescimento da Olist é a logística e a esperiência regional.

7.840 pedidos atrasados (7.9% do total).
Os estados com maior taxa de atraso - AL (23%), MA (19%), RR (11%) - concentram
as piores notas de satisfação, confirmando correlação direta entre atraso e insatisfação. RJ se destaca como risco estratégico: alto volume (12.791 pedidos) com nota abaixo da média.

## Indicadores Chave de Desempenho - KPIs

In [39]:
# Pegando último mês completo (agosto/2018)
ultimo_mes = crescimento_limpo[crescimento_limpo['anomes_referencia'] == 201808]

print('=== KPIs - Último mês disponível (Ago/2018) ===')
print(f"Volume de pedidos:  {ultimo_mes['volume_pedidos'].values[0]:,.0f}")
print(f"Faturamento:        R${ultimo_mes['faturamento_total'].values[0]:,.0f}")
print(f"Ticket médio:       R${ultimo_mes['ticket_medio'].values[0]:,.0f}")

print('\n=== Performance de Entrega ===')
total = performance['volume_pedidos'].sum()
for _, row in performance.iterrows():
    print(f"{row['performance_entrega']}: {row['volume_pedidos']/total*100:.1f}%")

print('\n=== Distribuição de Notas ===')
total_scores = scores['quantidade'].sum()
for _, row in scores.iterrows():
    print(f"{row['nota']}: {row['quantidade']/total_scores*100:.1f}%")

=== KPIs - Último mês disponível (Ago/2018) ===
Volume de pedidos:  6,489
Faturamento:        R$996,974
Ticket médio:       R$154

=== Performance de Entrega ===
Em Transito: 1.8%
Atrasado: 7.9%
No Prazo: 90.3%

=== Distribuição de Notas ===
Boa: 27.5%
Nao Avaliado: 0.7%
Otima: 57.8%
Pessima: 10.8%
Ruim: 3.2%


## Recomendação

Através dos dados apresentados, fica claro que a empresa está em fase de expansão: o crescimento é consistente e o ticket médio acompanha o crescimento da receita.

Porém, analisando a base, percebemos um problema no processo de entrega. Por mais que 58% dos clientes tenham avaliado com nota 5, 11% deram nota 1 e, cruzando os dados, foi possível identificar que a insatisfação está concentrada nos pedidos entregues fora do prazo combinado.


| KPI | Valor Atual | Meta 2019 |
|-----|-------------|-----------|
| Volume mensal de pedidos | ~6.500 | +15% (~7.500) |
| Receita mensal | ~RS1M | +20% (~RS1.2M) |
| Ticket médio | RS154 | +15% (~RS175) |
| % pedidos no prazo | 90.3% | 95% |
| % clientes nota Ótima | 57.8% | 63% |
| % clientes nota Péssima | 10.8% | 8% |


**Recomendação:** analisar os pedidos com atraso por região para identificar se o problema é concentrado em rotas específicas e priorizar a otimização logística nessas áreas, o que deve impactar diretamente a meta de satisfação e pontualidade para 2019.